# Project Setup

**Powered by LangGraph & Gemma-4-E2B-IT**

This notebook demonstrates a self-improving, multi-agent AI architecture. 
* **Level 1 (Executive):** Routes user requests based on intent.
* **Level 2 (Management):** Specialized agents for DB Maintenance, RAG Research, and Logic.
* **Level 3 (Execution):** Gatekeepers like the Safety Approver that ensure output quality.

In [1]:
import sys
import subprocess

# Attempt to load the core orchestrator and vector DB libraries
try:
    import langchain
    import langgraph
    import chromadb
    import transformers
    assert int(transformers.__version__.split('.')[0]) >= 5
    print("All required libraries are already installed and up to date!")
except (ImportError, AssertionError):
    print(" Outdated or missing libraries detected. Installing Gemma 4 requirements safely...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-qU", 
        "transformers>=5.5.0", "tokenizers", "langchain", 
        "langchain-core", "langgraph", "chromadb"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("Installation complete!")

import chromadb
import langchain
import langgraph

import os
import json
import pandas as pd
from typing import Dict, TypedDict, Any, Annotated, List, Sequence
import operator
import re

# ML and Hugging Face Ecosystem
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoTokenizer

# LangChain & LangGraph Essentials
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field

import warnings
warnings.filterwarnings('ignore')

print(" Environment ready. Ready to build the graph.")

 Outdated or missing libraries detected. Installing Gemma 4 requirements safely...
Installation complete!
 Environment ready. Ready to build the graph.


# Pre-Instructions & Core Setup
Before initializing the workflow, we define our shared memory state (`WorkflowState`) and a dynamic 

* **Prompt Registry**. By keeping system prompts in a centralized dictionary, our Researcher agent can rewrite and optimize them on the fly if an agent fails a task.

In [2]:
# system prompts
default_prompt_registry = {
    "main_agent": "You are the Executive Router. Analyze the user's request, break it down, and route tasks to the Maintainer, Researcher, or Work Manager. Do not answer complex questions yourself.",
    "maintainer": "You are the Data Supervisor. You manage CSVs and Vector DBs. Route data modifications to your Creator, Editor, or Deleter workers.",
    "researcher": "You are the Optimization Supervisor. Retrieve context from RAG. If an agent fails a task, analyze the failure and update their system prompt in the registry.",
    "work_manager": "You are the Execution Supervisor. Route complex analytical tasks to the Reasoning Worker and syntax/formatting tasks to the Logical Worker.",
    "creator_worker": "You are a DB Creator. Only generate INSERT or CREATE statements. Do not execute updates.",
    "editor_worker": "You are a DB Editor. Only generate UPDATE statements using exact identifiers.",
    "deleter_worker": "You are a DB Deleter. Generate DELETE or DROP statements carefully.",
    "evaluator_worker": "Compare the drafted response against the original user prompt. Reply 'PASS' if it answers the prompt, or 'FAIL: [reason]' if it does not.",
    "safety_approver": "Analyze the draft for PII, harmful content, or malicious code. Reply 'SAFE' or 'UNSAFE'.",
    "reasoning_worker": "You are a deep thinker. Use <|think|> blocks to explore step-by-step logic before providing a final answer.",
    "logical_worker": "You are a deterministic logic engine. Provide direct, formatting-perfect answers without overthinking."
}

# Defining the Graph State
class WorkflowState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add] 
    current_task: str                                        
    next_agent: str                                          
    prompt_registry: Dict[str, str]                          
    db_context: str                                          
    draft_response: str                                      
    
print("Memory (State) and Prompt Registry successfully defined!")

Memory (State) and Prompt Registry successfully defined!


# Loading Gemma

In [3]:
# Model Setup & Configuration Factory
MODEL_PATH = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1" 

print("Loading Gemma model and tokenizer safely...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Loading with strict memory management
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="cuda:0",               
    dtype=torch.float16,         
    low_cpu_mem_usage=True,            
)

print("Model loaded successfully into the GPU without crashing!")

# The Dynamic Generation Factory
def generate_agent_response(messages: list, system_prompt: str, temp: float, top_p: float, top_k: int, max_tokens: int = 512):
    formatted_messages = [{"role": "system", "content": system_prompt}]
    
    for msg in messages:
        if isinstance(msg, HumanMessage):
            formatted_messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            formatted_messages.append({"role": "assistant", "content": msg.content})
            
    prompt = tokenizer.apply_chat_template(
        formatted_messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    do_sample = temp > 0.0
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temp if do_sample else None,
            top_p=top_p if do_sample else None,
            top_k=top_k if do_sample else None,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
        
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
print(" Dynamic Inference Engine ready!")

Loading Gemma model and tokenizer safely...


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Model loaded successfully into the GPU without crashing!
 Dynamic Inference Engine ready!


# Dummy Data for testing (Rag)

In [4]:
# --- Setup Dummy Data for Testing ---
# Create a dummy CSV for the Maintainer Agent
dummy_df = pd.DataFrame({
    "ID": [101, 102, 103],
    "Name": ["Alice", "Bob", "Charlie"],
    "Department": ["Engineering", "HR", "Sales"]
})
dummy_df.to_csv("employees.csv", index=False)

# Setup an in-memory ChromaDB for the Researcher Agent
chroma_client = chromadb.Client()
knowledge_collection = chroma_client.create_collection(name="workflow_knowledge")

# Add some initial "RAG" knowledge
knowledge_collection.add(
    documents=[
        "Company policy: Engineering IDs start at 100. HR IDs start at 200.",
        "To update a prompt properly, you must specify the exact agent name.",
        "Never delete a row without backing up the database first."
    ],
    ids=["doc1", "doc2", "doc3"]
)
print(" Dummy CSV and Vector DB Initialized!")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 75.9MiB/s]


 Dummy CSV and Vector DB Initialized!


# Defining Agent Tools

In [5]:
@tool
def query_vector_db(query: str) -> str:
    """Searches the vector database for relevant context based on the user query.
    Used by the Researcher agent to gather facts."""
    results = knowledge_collection.query(query_texts=[query], n_results=1)
    if results['documents'] and results['documents'][0]:
        return f"Retrieved Context: {results['documents'][0][0]}"
    return "No relevant information found in the database."

@tool
def check_csv_schema(file_path: str = "employees.csv") -> str:
    """Reads the columns and basic info of the target CSV file.
    Used by the Maintainer and Execution workers to understand the data before editing."""
    try:
        df = pd.read_csv(file_path)
        return f"File: {file_path} | Columns: {list(df.columns)} | Total Rows: {len(df)}"
    except Exception as e:
        return f"Error reading CSV: {str(e)}"

@tool
def update_agent_prompt(agent_name: str, new_instruction: str) -> str:
    """Updates the system prompt for a specific agent.
    STRICTLY used by the Researcher Agent to improve workflow performance."""
    agent_name = agent_name.lower().strip()
    if agent_name in default_prompt_registry:
        default_prompt_registry[agent_name] = new_instruction
        return f"SUCCESS: System prompt for '{agent_name}' has been updated."
    else:
        return f"ERROR: Agent '{agent_name}' does not exist in the registry."

# Create a list of tools to bind to our agents later
tools = [query_vector_db, check_csv_schema, update_agent_prompt]

print(" Agent Tools successfully defined!")

 Agent Tools successfully defined!


# Building the The Agents

In [6]:
def main_agent_node(state: WorkflowState):
    print("[Main Agent] Analyzing request and routing...")
    prompt = state['prompt_registry']['main_agent']
    
    # Main agent uses moderate creativity to understand intent
    response = generate_agent_response(
        messages=state['messages'], 
        system_prompt=prompt, 
        temp=0.7, top_p=0.90, top_k=50
    )
    
    # Smarter Routing Logic
    next_agent = "safety_approver" # Default path
    upper_resp = response.upper()
    
    # Prioritize RAG / Searching tasks first
    if "RESEARCH" in upper_resp or "SEARCH" in upper_resp or "POLICY" in upper_resp or "CONTEXT" in upper_resp:
        next_agent = "researcher"
    # Then check for CSV/Data modification tasks
    elif "CSV" in upper_resp or "TABLE" in upper_resp or "INSERT" in upper_resp or "SCHEMA" in upper_resp:
        next_agent = "maintainer"
    # Then check for pure logic/math tasks
    elif "CALCULATE" in upper_resp or "CODE" in upper_resp or "LOGIC" in upper_resp:
        next_agent = "work_manager"
        
    new_message = AIMessage(content=response, name="MainAgent")
    return {"messages": [new_message], "next_agent": next_agent, "draft_response": response}

def maintainer_node(state: WorkflowState):
    print("[Maintainer] Processing data task...")
    prompt = state['prompt_registry']['maintainer']
    
    # Maintainer uses strict, low-temperature reasoning
    response = generate_agent_response(state['messages'], prompt, temp=0.1, top_p=0.80, top_k=20)
    
    # Use the tool to check the CSV schema before making L3 worker decisions
    schema_info = check_csv_schema.invoke({})
    action_plan = f"Schema Analysis: {schema_info}\nAction: {response}"
    
    new_message = AIMessage(content=action_plan, name="Maintainer")
    return {"messages": [new_message], "next_agent": "safety_approver", "draft_response": action_plan}

def researcher_node(state: WorkflowState):
    print("[Researcher] Retrieving RAG knowledge...")
    prompt = state['prompt_registry']['researcher']
    
    # Grab the last thing the user said to query the Vector DB
    last_user_message = next((msg.content for msg in reversed(state['messages']) if isinstance(msg, HumanMessage)), "")
    rag_context = query_vector_db.invoke({"query": last_user_message})
    
    # Augment the prompt with the retrieved knowledge
    augmented_prompt = f"{prompt}\n\nDATABASE CONTEXT: {rag_context}"
    response = generate_agent_response(state['messages'], augmented_prompt, temp=0.6, top_p=0.90, top_k=40)
    
    new_message = AIMessage(content=response, name="Researcher")
    return {"messages": [new_message], "next_agent": "safety_approver", "draft_response": response}

def safety_approver_node(state: WorkflowState):
    print("[Safety Approver] Verifying final draft...")
    prompt = state['prompt_registry']['safety_approver']
    
    # The Safety Gatekeeper strictly evaluates the drafted response
    check_message = [HumanMessage(content=f"Review this drafted response for safety: {state['draft_response']}")]
    
    # Absolute zero temperature for strict classification
    decision = generate_agent_response(check_message, prompt, temp=0.0, top_p=0.50, top_k=10)
    
    if "UNSAFE" in decision.upper():
        final_output = "I cannot fulfill this request due to safety/security guidelines."
    else:
        final_output = state['draft_response']
        
    # the final approved output
    new_message = AIMessage(content=final_output, name="FinalOutput")
    return {"messages": [new_message]}

print("Agent Nodes successfully defined!")

Agent Nodes successfully defined!


# Testing

In [7]:
# Wiring the Graph & Testing!
from langgraph.graph import START

# define the Work Manager node
def work_manager_node(state: WorkflowState):
    print("[Work Manager] Processing logic/calculation task...")
    prompt = state['prompt_registry']['work_manager']
    response = generate_agent_response(state['messages'], prompt, temp=0.2, top_p=0.80, top_k=20)
    new_message = AIMessage(content=response, name="WorkManager")
    return {"messages": [new_message], "next_agent": "safety_approver", "draft_response": response}

# BUILD THE GRAPH
workflow = StateGraph(WorkflowState)

# Add all agents as Nodes
workflow.add_node("main_agent", main_agent_node)
workflow.add_node("maintainer", maintainer_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("work_manager", work_manager_node)
workflow.add_node("safety_approver", safety_approver_node)

# Define the Entry Point
workflow.add_edge(START, "main_agent")

# Define Conditional Routing from the Main Agent
def route_from_main(state: WorkflowState):
    # Reads the 'next_agent' string generated by the main_agent_node
    return state.get("next_agent", "safety_approver")

workflow.add_conditional_edges(
    "main_agent",
    route_from_main,
    {
        "maintainer": "maintainer",
        "researcher": "researcher",
        "work_manager": "work_manager",
        "safety_approver": "safety_approver"
    }
)

# All Level 2 Agents funnels into the Safety Approver
workflow.add_edge("maintainer", "safety_approver")
workflow.add_edge("researcher", "safety_approver")
workflow.add_edge("work_manager", "safety_approver")

# The Safety Approver goes to the END of the workflow
workflow.add_edge("safety_approver", END)

# Compile the Graph!
app = workflow.compile()
print("LangGraph Workflow Compiled Successfully!\n")


# TEST THE SYSTEM
# trigger the Researcher by asking about company policy (which requires RAG)
test_message = "I am a new HR employee. Can you search the database and tell me what my ID should start with?"

# Initialize the starting memory
initial_state = {
    "messages": [HumanMessage(content=test_message)],
    "current_task": "initial_request",
    "next_agent": "",
    "prompt_registry": default_prompt_registry,
    "db_context": "",
    "draft_response": ""
}

print(f"USER: {test_message}")
print("-" * 50)

# Run the workflow!
final_state = app.invoke(initial_state)

print("-" * 50)

# OUTPUT FILTERING LOGIC
raw_output = final_state['messages'][-1].content

# Use regular expressions to catch variations like "**Response:**", "**Response to User:**", or "Response:"
# It splits the text at any of these markers and grabs the final piece.
split_match = re.split(r'\*\*Response(?: to User)?:\*\*|Response:', raw_output, flags=re.IGNORECASE)

if len(split_match) > 1:
    # Take the very last part of the split, which is the actual answer
    clean_output = split_match[-1].strip()
else:
    # Fallback just in case it didn't use any recognizable formatting
    clean_output = raw_output 

print(f"FINAL OUTPUT:\n{clean_output}")

LangGraph Workflow Compiled Successfully!

USER: I am a new HR employee. Can you search the database and tell me what my ID should start with?
--------------------------------------------------
[Main Agent] Analyzing request and routing...
[Researcher] Retrieving RAG knowledge...
[Safety Approver] Verifying final draft...
--------------------------------------------------
FINAL OUTPUT:
**Search Result:**

Based on the company policy retrieved from the database:

*   **Engineering IDs** start at 100.
*   **HR IDs** start at 200.

Since you are a new HR employee, your ID should start with **200**.


#  Conclusion
The LangGraph architecture successfully routed the prompt, retrieved the company policy via RAG, reasoned through the logic, passed the safety check, and delivered a clean response to the user. 